# Olist E-commerce - Data Preparation & Merging Pipeline

This notebook handles the ingestion of raw datasets, executes data quality audits (missing values and duplicates), and structures the two main relational tables required for business analysis: `customer_sales` and `product_sales`.

In [1]:
# ==================================================
# 1. Setup - Imports & Project Paths
# ==================================================
import pandas as pd
from pathlib import Path

# Define clean structural paths for the data pipeline
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

# Ensure the processed directory exists safely
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# ==================================================
# 2. Ingestion - Load Raw Datasets
# ==================================================

# Load all required relational tables from the raw directory
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
category_translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

print("All raw datasets loaded successfully!")

All raw datasets loaded successfully!


In [3]:
# ==================================================
# 3. Data Quality Audit - Comprehensive Report
# ==================================================

datasets = {
    "customers": customers,
    "orders": orders,
    "items": items,
    "payments": payments,
    "products": products,
    "category_translation": category_translation
}

# Define strict width for a unified summary table
PANEL_WIDTH = 78
print("=" * PANEL_WIDTH)
print(f"{'INITIAL DATA QUALITY AUDIT REPORT':^{PANEL_WIDTH}}")
print("=" * PANEL_WIDTH)
print(f"{'Dataset Name':<25}{'Rows':>12}{'Columns':>10}{'Null Columns':>16}{'Duplicates':>12}")
print("-" * PANEL_WIDTH)

# Generate unified metrics per dataset
for name, df in datasets.items():
    rows, cols = df.shape
    null_cols_count = (df.isnull().sum() > 0).sum()
    duplicate_rows = df.duplicated().sum()
    
    print(f"{name:<25}{rows:>12,}{cols:>10}{null_cols_count:>16}{duplicate_rows:>12,}")

print("=" * PANEL_WIDTH)

# Detailed breakdown for missing values (only prints if missing data exists)
print("\n" + "-" * 40)
print("Detailed Missing Values Breakdown:")
print("-" * 40)
for name, df in datasets.items():
    missing_series = df.isnull().sum()
    missing_series = missing_series[missing_series > 0]
    if not missing_series.empty:
        print(f"\n[{name.upper()}]")
        for col, val in missing_series.items():
            print(f" - {col:<32} -> {val:,} missing rows")

                      INITIAL DATA QUALITY AUDIT REPORT                       
Dataset Name                     Rows   Columns    Null Columns  Duplicates
------------------------------------------------------------------------------
customers                      99,441         5               0           0
orders                         99,441         8               3           0
items                         112,650         7               0           0
payments                      103,886         5               0           0
products                       32,951         9               8           0
category_translation               71         2               0           0

----------------------------------------
Detailed Missing Values Breakdown:
----------------------------------------

[ORDERS]
 - order_approved_at                -> 160 missing rows
 - order_delivered_carrier_date     -> 1,783 missing rows
 - order_delivered_customer_date    -> 2,965 missing rows

[PRODUCTS

## 4. Building the Customer Sales Dataset
This pipeline consolidates customer profiles, order contexts, and transaction values to support revenue, AOV, and customer behavior analyses.

In [5]:
# ==================================================
# 4.1 Pipeline - Customer Sales Compilation
# ==================================================

# Step 1: Link orders to their respective customer demographic profile
customer_sales = pd.merge(orders, customers, on="customer_id", how="left")

# Step 2: Enrich the dataset with payment and transaction details
customer_sales = pd.merge(customer_sales, payments, on="order_id", how="left")

# Step 3: Export final structured table to the processed folder
customer_sales.to_csv(PROCESSED_DIR / "customer_sales.csv", index=False)

# Executive Verification
print("=" * 50)
print(f"{'CUSTOMER SALES DATASET EXPORTED':^50}")
print("=" * 50)
print(f"{'File Name:':<20}{'customer_sales.csv':>26}")
print(f"{'Final Shape:':<20}{str(customer_sales.shape):>26}")
print("=" * 50)

         CUSTOMER SALES DATASET EXPORTED          
File Name:                  customer_sales.csv
Final Shape:                      (103887, 16)


## 5. Building the Product Sales Dataset
This pipeline focuses on item-level logs, joining products with their categories, applying translations, and cleaning text variables for item and category insights.

In [6]:
# ==================================================
# 5.1 Pipeline - Product Sales Compilation
# ==================================================

# Step 1: Map items to their technical product specifications
items_products = pd.merge(items, products, on="product_id", how="left")

# Step 2: Connect product items back to their root order context
product_sales = pd.merge(orders, items_products, on="order_id", how="left")

# Step 3: Apply English translations to category names
product_sales = pd.merge(product_sales, category_translation, on="product_category_name", how="left")

# Step 4: Handle missing values in translations natively to protect downstream analysis
product_sales["product_category_name_english"] = product_sales["product_category_name_english"].fillna("Unknown")

# Step 5: Export final clean dataset to the processed folder
product_sales.to_csv(PROCESSED_DIR / "product_sales.csv", index=False)

# Executive Verification
print("=" * 50)
print(f"{'PRODUCT SALES DATASET EXPORTED':^50}")
print("=" * 50)
print(f"{'File Name:':<20}{'product_sales.csv':>26}")
print(f"{'Final Shape:':<20}{str(product_sales.shape):>26}")
print("=" * 50)

          PRODUCT SALES DATASET EXPORTED          
File Name:                   product_sales.csv
Final Shape:                      (113425, 23)
